# الخطوة 2: الإعدادات ونمط الـ Singleton (`core/config.py`)

---

## 💡 المفهوم البرمجي: نمط الـ Singleton ولماذا نستخدمه؟

في تطبيقنا، نريد قراءة ملف `config.json` **مرة واحدة فقط** من القرص وحفظه في الذاكرة.

إذا قام أي جزء من التطبيق بطلب الإعدادات (`Config()`)، يجب أن يحصل على **نفس النسخة (Instance)** في الذاكرة بدلاً من إعادة فتح الملف وقراءته من جديد.

في بايثون، نطبق ذلك بالتحكم في دالة البناء السحرية `__new__` (التي تُنشئ الكائن قبل استدعاء `__init__`).

---

## 📊 مخطط تدفق العمليات (Flowchart)

```mermaid
graph TD
    A[1. Call Config] --> B{Instance exists?}
    B -- No --> C[2. Create instance]
    C --> D[3. Load config]
    D --> E{File exists?}
    E -- No --> F[Raise ConfigError]
    E -- Yes --> G[4. Parse JSON]
    G --> H{Valid JSON?}
    H -- No --> I[Raise ConfigError]
    H -- Yes --> J[5. Store in _data]
    J --> K[6. Return instance]
    B -- Yes --> K

    style C fill:#f9f,stroke:#333,stroke-width:2px
    style D fill:#bbf,stroke:#333,stroke-width:2px
    style K fill:#bfb,stroke:#333,stroke-width:2px
```

**شرح المخطط:**

| المرحلة | الوصف |
|---|---|
| **1. Call `Config()`** | أول استدعاء للكلاس |
| **2. Create instance** | `__new__` يتحقق: هل فيه نسخة موجودة؟ لو لأ → ينشئ واحدة |
| **3. Load config** | استدعاء `_load()` لقراءة الملف |
| **4. Parse JSON** | فحص وجود الملف وصحة صياغة JSON |
| **5. Store in `_data`** | تخزين القيم في الذاكرة |
| **6. Return instance** | إرجاع نفس النسخة لأي استدعاء لاحق (بدون إعادة تحميل) |

---

## ✅ التحقق من الخطوة 2

تأكد من وجود ملف `config.json` في المجلد الرئيسي، ثم شغّل هذا الأمر في الـ Terminal:

```bash
python3 -c "from core.config import Config; c1 = Config(); c2 = Config(); print(c1.get('app_name')); print('Is Singleton?', c1 is c2)"
```

**المتوقع أن يطبع:**

```
Forex Tracker
Is Singleton? True
```

إذا طبع لك ذلك، أخبرني فوراً لننتقل إلى **الخطوة 3: نظام التسجيل (`core/logger.py`)**.

In [ ]:
from core.config import Config

print("Success!")
c = Config()
print(c.get("app_name"))
print(c.get("version"))

Success!
Forex Tracker
0.1.0


In [2]:
c.get("log_file", "./logs/app.log")

'./logs/app.log'

# الخطوة 3: نظام التسجيل الموحّد (`core/logger.py`)

---

## 💡 المفهوم البرمجي: لماذا لا نستخدم `print()` ونحتاج `logging`؟

في الأنظمة الحقيقية والـ **Trading Systems**:

- الـ `print()` مجرد طباعة على الشاشة وتنتهي بمجرد إغلاق الـ Terminal.

- الـ `logging` يتيح لنا:
  - **حفظ السجلات في ملف** (`logs/app.log`) لمراجعة ما حدث في أي وقت (Auditing).
  - **تحديد مستويات الأهمية** (INFO, WARNING, ERROR, DEBUG).
  - **إضافة طابع زمني دقيق** (timestamp) لكل حدث.

---

## 📊 مخطط تدفق العمليات (Flowchart)

```mermaid
graph TD
    A[1. Call setup_logger] --> B{Logger has handlers?}
    B -- Yes --> C[Return existing logger]
    B -- No --> D[2. Set level INFO]
    D --> E[3. Get log path from Config]
    E --> F[4. Create logs folder]
    F --> G[5. Build formatter]
    G --> H[6. Add FileHandler]
    H --> I[7. Add ConsoleHandler]
    I --> J[8. Return logger]

    style D fill:#bbf,stroke:#333,stroke-width:2px
    style H fill:#f9f,stroke:#333,stroke-width:2px
    style J fill:#bfb,stroke:#333,stroke-width:2px
```

**شرح المخطط:**

| المرحلة | الوصف |
|---|---|
| **1. Call `setup_logger()`** | أول استدعاء لتهيئة الـ Logger |
| **2. Set level INFO** | تحديد الحد الأدنى لمستوى الأهمية |
| **3. Get log path** | قراءة مسار ملف السجلات من `Config` |
| **4. Create logs folder** | إنشاء المجلد تلقائياً لو مش موجود |
| **5. Build formatter** | تحديد شكل السطر (تاريخ + مستوى + رسالة) |
| **6. Add FileHandler** | إرسال السجلات إلى الملف |
| **7. Add ConsoleHandler** | إرسال السجلات إلى الشاشة |
| **8. Return logger** | إرجاع الـ Logger جاهز للاستخدام |

---

---

## 🔍 شرح المكونات الأساسية

| المكوّن | الدور |
|---|---|
| `logging.getLogger(name)` | إنشاء/جلب Logger باسم محدد |
| `logger.handlers` | فحص لتجنب تكرار الـ Handlers عند إعادة الاستدعاء |
| `logger.setLevel(INFO)` | تحديد مستوى الأهمية الأدنى |
| `config.get("log_file", ...)` | قراءة مسار الملف من الإعدادات مع قيمة افتراضية |
| `Path(...).parent.mkdir(...)` | إنشاء مجلد `logs/` تلقائياً |
| `Formatter` | تحديد شكل السطر: `[تاريخ] [مستوى] [اسم]: رسالة` |
| `FileHandler` | كتابة السجلات في الملف |
| `StreamHandler` | طباعة السجلات على الشاشة |

---

## ✅ التحقق من الخطوة 3

شغّل الأمر التالي في الـ Terminal:

```bash
python3 -c "from core.logger import logger; logger.info('Logger is working successfully!')"
```

**المتوقع:**

- ستظهر لك الرسالة على شاشة الـ Terminal وبها التاريخ والوقت ومستوى الأهمية `[INFO]`.

- سيتم إنشاء ملف داخل مجلد `logs/app.log` وتجد نفس السطر مكتوباً فيه.

In [5]:
import logging
name = "MyApp"
logger = logging.getLogger(name)

In [12]:
print(list(logger.manager.loggerDict))

['concurrent.futures', 'concurrent', 'asyncio', 'tornado.access', 'tornado', 'tornado.application', 'tornado.general', 'traitlets', 'stack_data.serializing', 'stack_data', 'prompt_toolkit.buffer', 'prompt_toolkit', 'parso.python.diff', 'parso.python', 'parso', 'parso.cache', 'ipykernel.iostream', 'ipykernel', 'Comm', 'ipykernel.comm', 'IPKernelApp', 'MyApp']


In [1]:
from core.logger import logger
logger.info('Logger is working successfully!')

[2026-10-11 01:00:35] [INFO] [forex_tracker]: Logger is working successfully!


# 4️⃣ الخطوة 4: الـ Decorator الأساسي (`utils/decorators.py`)

## 🔍 المفهوم البرمجي: كيف يعمل الـ Decorator في بايثون؟

الـ **Decorator** هو دالة تأخذ دالة كمدخل، وتُرجع دالة جديدة (تسمى **Wrapper**).

الـ Wrapper هذا ينفذ:

1. **كوداً قبل** استدعاء الدالة الأصلية (تسجيل بداية الوقت).
2. **استدعاء الدالة الأصلية**.
3. **كوداً بعدها** (حساب الفرق الزمني وتسجيله عبر `logger.info`).

استخدام `@functools.wraps(func)` **ضروري جداً** لكي تحتفظ الدالة الأصلية باسمها ومستنداتها (`__name__`, `__doc__`).

---

## 📊 مخطط تدفق العمليات (Flowchart)

```mermaid
graph TD
    A[1. Call decorated function] --> B[2. wrapper starts]
    B --> C[3. start_time = perf_counter]
    C --> D[4. Call original func]
    D --> E[5. Get result]
    E --> F[6. duration = end - start]
    F --> G[7. logger.info duration]
    G --> H[8. return result]

    style C fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#f9f,stroke:#333,stroke-width:2px
    style H fill:#bfb,stroke:#333,stroke-width:2px
```

**شرح المخطط:**

| المرحلة | الوصف |
|---|---|
| **1. Call decorated function** | استدعاء الدالة بعد تزيينها بـ `@timer` |
| **2. wrapper starts** | الدالة الجديدة (Wrapper) تبدأ التنفيذ |
| **3. start_time** | تسجيل لحظة البداية بدقة عالية |
| **4. Call original func** | استدعاء الدالة الأصلية بكل معاملاتها |
| **5. Get result** | استلام الناتج |
| **6. duration** | حساب الفرق الزمني |
| **7. logger.info** | تسجيل المدة في الـ Logger |
| **8. return result** | إرجاع الناتج كما هو (شفافية كاملة) |


## 🔍 شرح المكونات الأساسية

| المكوّن | الدور |
|---|---|
| `functools.wraps(func)` | يحافظ على `__name__`, `__doc__`, `__module__` للدالة الأصلية |
| `*args, **kwargs` | يستقبل أي عدد من المعاملات (Positional & Keyword) |
| `time.perf_counter()` | ساعة عالية الدقة لقياس الفترات الزمنية |
| `duration:.4f` | تنسيق الرقم إلى 4 خانات عشرية |
| `return result` | إرجاع ناتج الدالة الأصلي كما هو |

---

## 🎯 لماذا `time.perf_counter()` وليس `time.time()`؟

| الدالة | الاستخدام الأمثل |
|---|---|
| `time.time()` | الوقت الفعلي (Wall Clock) — يتأثر بتغيير ساعة النظام |
| `time.perf_counter()` | **قياس الفترات الزمنية** — أعلى دقة متاحة، لا يتأثر بتغيير الساعة |

للـ Benchmarking دايماً استخدم `perf_counter`.

---

## 🧪 التحقق من عمل `@timer`

شغّل هذا الأمر السريع من الـ Terminal:

```bash
python3 -c "
from utils.decorators import timer
import time

@timer
def sample_task():
    time.sleep(0.05)
    return 'done'

print('Result:', sample_task())
"
```

**المتوقع:**

- تطبع رسالة من الـ Logger توضح اسم الدالة والوقت المستغرق (مثلاً: `Execution: 'sample_task' took 0.050...s`).
- تطبع الناتج: `Result: done`.

---

## 🔬 اختبار إضافي: التحقق من `functools.wraps`

لتتأكد بنفسك إن `functools.wraps` شغال صح:

```bash
python3 -c "
from utils.decorators import timer

@timer
def my_func():
    '''وثيقة الدالة الأصلية'''
    pass

print('Name:', my_func.__name__)
print('Doc:', my_func.__doc__)
"
```

**المتوقع:**

```
Name: my_func
Doc: وثيقة الدالة الأصلية
```

> لو حذفت `@functools.wraps(func)` هتلاقي `Name: wrapper` و `Doc: None` — وده بيكسر التوثيق والـ Debugging.

---

## 📂 موقع الملف في الهيكل

```
project_root/
├── core/
│   ├── config.py
│   ├── logger.py
│   └── exceptions.py
├── utils/
│   └── decorators.py      ← ✅ الملف ده
├── logs/
│   └── app.log
└── config.json
```

---

نفّذ الخطوة وتأكد منها، ثم اكتب لي **next** لننتقل إلى **الخطوة 5: عقد التخزين المجرد (`storage/base.py`) والـ `storage/memory_storage.py`**.

In [ ]:
from utils.decorators import timer
import time

@timer
def sample_task():
    time.sleep(0.05)
    return 'done'

print('Result:', sample_task())


[2026-10-11 11:22:57] [INFO] [forex_tracker]: sample_task took 0.0000 seconds to execute.


Result: done


In [ ]:
from utils.decorators import timer
import time

@timer
def sample_task():
    return 'done'

print('Result:', sample_task())


# 5️⃣ الخطوة 5: عقد التخزين المجرد (`storage/base.py`) والـ In-Memory Storage (`storage/memory_storage.py`)

---

## 💡 المفهوم المعماري: لماذا نحتاج `Storage` ABC؟

في خطتنا وقواعدنا الذهبية:

- **في Sprint 0**: نريد تخزين بيانات في الذاكرة مؤقتاً لاختبار النظام بدون الحاجة لقواعد بيانات (`MemoryStorage`).
- **في Sprint 1**: سنبني `SQLiteStorage` لحفظ الشموع (Candles).
- **في Sprint 2**: سنضيف دعم المستخدمين والجلسات.

لو جعلنا خدماتنا (`services`) تعتمد مباشرة على SQLite، سنقع في فخ **الارتباط الشديد (Tight Coupling)**، ولن نستطيع عمل Unit Tests سريعة ومعزولة بدون فتح وغلق ملفات داتابيز!

**الحل:**
نبني "عقداً" برمجياً يُسمى **Abstract Base Class (ABC)**.
الـ Service تقول: *"أنا أتعامل مع أي شيء ينفذ عقد Storage، ولا يهمني إن كان يكتب في الذاكرة أو في SQLite أو في Postgres"*.

---

## 📊 مخطط العلاقة بين الـ ABC والـ Implementation

```mermaid
graph TD
    A[Storage ABC] --> B[connect - abstract]
    A --> C[disconnect - abstract]
    A --> D[is_connected - abstract]
    
    E[MemoryStorage] -.implements.-> A
    F[SQLiteStorage - Sprint 1] -.implements.-> A
    G[PostgresStorage - Future] -.implements.-> A
    
    E --> E1[_connected: bool]
    E --> E2[_data: dict]
    
    style A fill:#f9f,stroke:#333,stroke-width:3px
    style E fill:#bfb,stroke:#333,stroke-width:2px
    style F fill:#bbf,stroke:#333,stroke-width:2px
    style G fill:#ddd,stroke:#333,stroke-width:2px
```

**شرح المخطط:**

| العنصر | الوصف |
|---|---|
| **Storage (ABC)** | العقد الذي يحدد 3 دوال إلزامية |
| **MemoryStorage** | التطبيق الحالي (Sprint 0) — في الذاكرة |
| **SQLiteStorage** | التطبيق القادم (Sprint 1) — على القرص |
| **PostgresStorage** | تطبيق مستقبلي محتمل |

أي كلاس يورث من `Storage` **لازم** ينفذ الدوال الثلاثة، وإلا Python هترفض إنشاء كائن منه.

---

## 📊 مخطط دورة حياة `MemoryStorage`

```mermaid
stateDiagram-v2
    [*] --> Disconnected: __init__
    Disconnected --> Connected: connect()
    Connected --> Disconnected: disconnect()
    Connected --> Connected: is_connected() = True
    Disconnected --> Disconnected: is_connected() = False
```

---

## 📝 الجزء الأول: `storage/base.py`

نطبق القاعدة الذهبية: **Storage يُبنى method by method**.
في Sprint 0 لسنا بحاجة لأي جداول عملات بعد، نريد فقط:

- دالة **للاتصال**.
- دالة **لغلق الاتصال**.
- دالة **لفحص الحالة**.

### 🔍 شرح المكونات

| المكوّن | الدور |
|---|---|
| `ABC` | الكلاس الأساسي لإنشاء Abstract Classes في Python |
| `@abstractmethod` | يمنع إنشاء كائن من `Storage` مباشرة، ويُلزم الورثة بتنفيذ الدالة |
| `-> None` / `-> bool` | Type Hints لتوضيح نوع القيمة المُرجعة |
| `pass` | لا يوجد تنفيذ — مجرد توقيع (Signature) |

> ⚠️ **ملاحظة**: لو حاولت `Storage()` مباشرة → `TypeError: Can't instantiate abstract class`.

---

### 🔍 شرح المكونات

| المكوّن | الدور |
|---|---|
| `_connected: bool` | علم (Flag) يتابع حالة الاتصال |
| `_data: dict` | مكان تخزين البيانات فعلياً (فاضي حالياً في Sprint 0) |
| `connect()` | يحوّل الحالة إلى `True` ويسجل في الـ Logger |
| `disconnect()` | يحوّل الحالة إلى `False` **ويمسح البيانات** |
| `is_connected()` | يرجّع الحالة الحالية |

---

## 🎯 القيمة المعمارية لهذه الخطوة

| الفائدة | الشرح |
|---|---|
| **Loose Coupling** | الخدمات تعتمد على `Storage` مش على تطبيق محدد |
| **Testability** | اختبارات سريعة بدون فتح ملفات داتابيز |
| **Extensibility** | إضافة `SQLiteStorage` غداً بدون تعديل الخدمات |
| **Contract Enforcement** | Python هترفض أي تطبيق ناقص للدوال المطلوبة |
| **Documentation** | الـ ABC نفسه وثيقة حية لما يجب على أي Storage فعله |

---

## 🧪 التحقق من عمل العقد والـ MemoryStorage

شغّل هذا الأمر السريع في الـ Terminal:

```bash
python3 -c "
from storage.memory_storage import MemoryStorage
storage = MemoryStorage()
print('Before connect:', storage.is_connected())
storage.connect()
print('After connect:', storage.is_connected())
storage.disconnect()
print('After disconnect:', storage.is_connected())
"
```

**المتوقع أن يطبع:**

```
Before connect: False
After connect: True
After disconnect: False
```

بالإضافة إلى رسالتين في الـ Logger:
```
[timestamp] [INFO] [forex_tracker]: MemoryStorage connected.
[timestamp] [INFO] [forex_tracker]: MemoryStorage disconnected.
```

---

## 🧪 اختبار إضافي: التحقق من إلزامية العقد

لو حاولت تعمل كلاس ناقص:

```bash
python3 -c "
from storage.base import Storage

class BrokenStorage(Storage):
    pass

BrokenStorage()
"
```

**المتوقع:**
```
TypeError: Can't instantiate abstract class BrokenStorage 
with abstract methods connect, disconnect, is_connected
```

ده **دليل قاطع** إن الـ ABC شغال صح وبيفرض العقد.

---

## 📂 موقع الملفات في الهيكل

```
project_root/
├── core/
│   ├── config.py
│   ├── logger.py
│   └── exceptions.py
├── utils/
│   └── decorators.py
├── storage/                    ← ✅ جديد
│   ├── __init__.py
│   ├── base.py                 ← العقد
│   └── memory_storage.py       ← التطبيق
├── logs/
│   └── app.log
└── config.json
```

---

نفّذ الخطوة وتأكد منها، ثم اكتب لي **next** لننتقل إلى **الخطوة 6 والأخيرة في Sprint 0**: كتابة الـ Unit Tests وتشغيلها وإعلان اكتمال Sprint 0 بنجاح! 🚀

In [3]:
from storage.memory_storage import MemoryStorage
storage = MemoryStorage()
print('Before connect:', storage.is_connected())
storage.connect()
print('After connect:', storage.is_connected())
storage.disconnect()
print('After disconnect:', storage.is_connected())

[2026-10-11 12:38:39] [INFO] [forex_tracker]: MemoryStorage connected.
[2026-10-11 12:38:39] [INFO] [forex_tracker]: MemoryStorage disconnected.


Before connect: False
After connect: True
After disconnect: False


# 6️⃣ الخطوة 6: ملف `tests/test_foundation.py`

---
## 💡 الفكرة المعمارية (لماذا نفعل ذلك؟)

- **ضمان سلامة البنية التحتية**: التأكد من أن المكونات الأساسية (الإعدادات، التخزين، والمؤقت) تعمل بشكل صحيح ومتناسق قبل الانتقال للمراحل المتقدمة.
- **منع التراجع (Regression Prevention)**: حماية الكود من التلف عند إضافة ميزات جديدة أو تعديل أجزاء أخرى من النظام مستقبلاً.
- **أتمتة الفحص**: تشغيل اختبارات الوحدة (Unit Tests) بضغطة زر واحدة لتوفير الوقت والجهد بدلاً من الفحص اليدوي.

---

## 📊 مخطط تدفق العمليات (Flowchart)

```mermaid
graph TD
    A[بداية تشغيل الاختبارات] --> B{test_config_singleton_and_values}
    B --> B1[assertIs: تطابق الكائن في الذاكرة]
    B1 --> B2[assertEqual: صحة القيم المقروءة]
    
    A --> C{test_memory_storage_lifecycle}
    C --> C1[assertFalse: التحقق من وضع عدم الاتصال]
    C1 --> C2[assertTrue: التحقق من نجاح الاتصال]
    
    A --> D{test_timer_decorator}
    D --> D1[assertEqual: التأكد أن المخرجات لم تتغير]
    
    B2 & C2 & D1 --> E[إعلان النتيجة: OK أو Failure]

    style B1 fill:#bbf,stroke:#333,stroke-width:2px
    style C1 fill:#f9f,stroke:#333,stroke-width:2px
    style D1 fill:#bfb,stroke:#333,stroke-width:2px
```

> ⚠️ يُرجى استخدام الرمز البرمجي بحذر.

---

## 🔍 دوال التحقق الفريدة المستخدمة (Assertion Methods)

استخدمنا **4 دوال فحص فريدة** من مكتبة `unittest` لأهداف محددة:

| الدالة | الغرض | مثال الاستخدام |
|---|---|---|
| `self.assertIs(a, b)` | للتحقق من أن المتغيرين يشيران إلى **نفس الكائن تماماً** في الذاكرة | اختبار نمط الـ Singleton لـ `Config` |
| `self.assertEqual(a, b)` | للتحقق من **تساوي القيم** المرتجعة | صحة نصوص الإعدادات، وسلامة مخرجات الدالة بعد تطبيق الـ Decorator |
| `self.assertTrue(x)` | للتأكد من أن الشرط صحيح وعائد بقيمة `True` | حالة التخزين بعد الـ `connect` |
| `self.assertFalse(x)` | للتأكد من أن الشرط خاطئ وعائد بقيمة `False` | حالة التخزين قبل الاتصال وبعد الـ `disconnect` |

---

## 📂 مكونات الهيكل

- **`tests/test_foundation.py`**: ملف فحص الوحدة الذي يحتوي على فئة الاختبار `TestFoundation` المنبثقة من `unittest.TestCase`.

In [9]:
from tests.test_foundation import TestFoundation

t = TestFoundation()

t.test_config_singleton_and_values()
t.test_memory_storage_lifecycle()
t.test_timer_decorator()

[2026-10-11 13:03:50] [INFO] [forex_tracker]: Config singleton and values test passed.
[2026-10-11 13:03:50] [INFO] [forex_tracker]: MemoryStorage connected.
[2026-10-11 13:03:50] [INFO] [forex_tracker]: MemoryStorage disconnected.
[2026-10-11 13:03:50] [INFO] [forex_tracker]: MemoryStorage lifecycle test passed.
[2026-10-11 13:03:50] [INFO] [forex_tracker]: sample_add took 0.1012 seconds to execute.
[2026-10-11 13:03:50] [INFO] [forex_tracker]: Timer decorator test passed.


# التحديث المعماري الكامل لطبقة الـ CLI

---

## 🎯 المبدأ المعماري: Command / Dispatch Pattern

بناءً على مبدأ **YAGNI** وتطبيق نمط **Command / Dispatch Pattern**، بحيث يكون كل ملف له مسؤولية محددة وواضحة (**Single Responsibility**).

---

## 📂 الهيكل النهائي المستهدف لـ CLI

```
forex_tracker/
├── main.py              # نقطة البداية (Entry Point) — للتشغيل المباشر
├── pyproject.toml       # تعريف أمر النظام forex
└── cli/
    ├── __init__.py      # يجعل المجلد حزمة Python
    ├── commands.py      # دوال تنفيذ الأوامر (Execution logic)
    └── parser.py        # تعريف الخيارات + نقطة تشغيل CLI main()
```

> ⚠️ **تأكد من حذف** `facade.py` و `__main__.py` من داخل مجلد `cli/` لتجنب التشتت والـ Over-engineering.

---

## 📊 مخطط تدفق تنفيذ الأوامر (Flowchart)

```mermaid
graph TD
    A[User: forex version] --> B[cli.parser:main]
    A2[User: python main.py version] --> B2[main.py]
    B --> C[sys.argv check]
    B2 --> C
    C -->|Empty| D[Print help & exit]
    C -->|Has args| E[build_parser]
    E --> F[parse_args]
    F --> G{Command matched?}
    G -->|version| H[cmd_version]
    G -->|status| I[cmd_status]
    G -->|Unknown| D
    H --> J[Print version from Config]
    I --> K[Print status + log to file]

    style B fill:#bbf,stroke:#333,stroke-width:2px
    style B2 fill:#bbf,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333,stroke-width:2px
    style H fill:#bfb,stroke:#333,stroke-width:2px
    style I fill:#bfb,stroke:#333,stroke-width:2px
```

**شرح المخطط:**

| المرحلة | الوصف |
|---|---|
| **نقطتا الدخول** | `main.py` (يدوي) و `cli.parser:main` (أمر النظام `forex`) |
| **build_parser** | يبني الـ Parser ويربط كل أمر بدالته |
| **parse_args** | يحلل مدخلات المستخدم |
| **Dispatch** | `set_defaults(func=...)` يحدد الدالة المطلوبة |
| **cmd_version / cmd_status** | التنفيذ الفعلي |

---

## 📊 مخطط العلاقة بين الملفات

```mermaid
graph LR
    A[main.py] -->|imports main| B[cli/parser.py]
    A2[forex entry] -->|cli.parser:main| B
    B -->|imports| C[cli/commands.py]
    B -->|imports| D[core/config.py]
    C -->|imports| D
    C -->|imports| E[core/logger.py]
    
    style A fill:#bbf,stroke:#333,stroke-width:2px
    style A2 fill:#bbf,stroke:#333,stroke-width:2px
    style B fill:#f9f,stroke:#333,stroke-width:2px
    style C fill:#bfb,stroke:#333,stroke-width:2px
```

**الاتجاه دايماً للأمام** — `main` → `parser` → `commands` → `core`. لا يوجد اعتماد عكسي.

---

## 1️⃣ ملف `cli/commands.py`

هذا الملف مسؤول **فقط** عن سلوك كل أمر (ماذا يفعل النظام عندما يطلب المستخدم أمراً معيناً).

### 🔍 شرح المكونات

| المكوّن | الدور |
|---|---|
| `cmd_version(args)` | يقرأ `app_name` و `version` من Config ويطبعهما |
| `cmd_status(args)` | يطبع الإعدادات الأساسية ويسجل الحدث في الـ Logger |
| `args` parameter | حتى لو مش مستخدم الآن، لازم يكون موجود للتوافق مع `set_defaults` |
| `config.get(key, default)` | قراءة آمنة مع قيمة افتراضية |

---

## 2️⃣ ملف `cli/parser.py`

هنا نستخدم **Subparsers** مع ربط كل أمر بدالته تلقائياً عبر `set_defaults(func=...)`، **وكذلك نضع دالة `main()` هنا** لأنها نقطة الدخول لأمر النظام `forex`.

### 🔍 شرح المكونات

| المكوّن | الدور |
|---|---|
| `argparse.ArgumentParser` | الكلاس الأساسي لتحليل المدخلات |
| `prog="forex"` | الاسم اللي يظهر في رسالة الـ Help |
| `subparsers` | يسمح بأوامر فرعية (`version`, `status`) |
| `set_defaults(func=...)` | **التريكة الذهبية** — تربط الأمر بالدالة مباشرة |
| `dest="command"` | يخزن اسم الأمر في `args.command` |
| **`main()`** | نقطة التشغيل الموحدة لأمر `forex` ومن `main.py` |

### 🎯 لماذا `set_defaults(func=...)` أفضل من `if/elif`؟

| الطريقة | المميزات | العيوب |
|---|---|---|
| `if/elif` | مفهومة للمبتدئين | تتناقض مع Open/Closed Principle |
| `set_defaults` | إضافة أمر = سطرين فقط | تحتاج فهم الـ Dispatch Pattern |

مع `set_defaults`، إضافة أمر جديد = **سطرين** (`add_parser` + `set_defaults`). مع `if/elif` = تعديل `main.py` كل مرة.

---

## 3️⃣ ملف `main.py` (في المجلد الرئيسي للمشروع)

نقطة البداية تكون **صغيرة جداً ونظيفة**، ووظيفتها الوحيدة هي تمرير التحكم إلى `cli.parser.main`:

### 🔍 شرح المكونات

| المكوّن | الدور |
|---|---|
| `from cli.parser import main` | استيراد نقطة التشغيل الموحدة |
| `if __name__ == "__main__"` | يضمن التنفيذ فقط عند التشغيل المباشر |
| **عدم تكرار المنطق** | كل المنطق في `cli/parser.py` — `main.py` مجرد واجهة |

> 💡 **تحسين معماري**: بدلاً من تكرار منطق `try/except` و `parse_args` في ملفين، نوحّده في `cli/parser.py:main()` ونخلي `main.py` مجرد سطرين. ده **DRY (Don't Repeat Yourself)** في أنقى صوره.

---

## 4️⃣ ملف `pyproject.toml` — ربط أمر النظام

لتفعيل أمر `forex` من أي مكان في البيئة الافتراضية:

```toml
[project.scripts]
forex = "cli.parser:main"
```

### 🔍 شرح المكوّن

| الجزء | المعنى |
|---|---|
| `forex` | اسم الأمر اللي هيتنفذ في الطرفية |
| `cli.parser` | مسار الموديول |
| `:main` | اسم الدالة داخل الموديول |

**كيف يشتغل؟** `pip install -e .` بيقرأ `[project.scripts]` وبيعمل سكريبت تنفيذي في `venv/bin/forex` بينادي `cli.parser.main()`.

---

## 📂 الهيكل النهائي الكامل للمشروع

```
forex_tracker/
├── main.py                     ← نقطة البداية (يدوي)
├── pyproject.toml              ← تعريف أمر forex
├── config.json
├── cli/
│   ├── __init__.py
│   ├── commands.py             ← منطق الأوامر
│   └── parser.py               ← تعريف الأوامر + main()
├── core/
│   ├── __init__.py
│   ├── config.py
│   ├── logger.py
│   └── exceptions.py
├── storage/
│   ├── __init__.py
│   ├── base.py
│   └── memory_storage.py
├── utils/
│   ├── __init__.py
│   └── decorators.py
├── tests/
│   ├── __init__.py
│   └── test_foundation.py
└── logs/
    └── app.log
```

---

## 🎯 القيمة المعمارية لهذه الطبقة

| المبدأ | التطبيق |
|---|---|
| **Single Responsibility** | كل ملف له دور واحد واضح |
| **Separation of Concerns** | Parsing منفصل عن Execution |
| **Open/Closed Principle** | إضافة أمر جديد لا تعدل الكود القديم |
| **DRY** | منطق التشغيل موحّد في `cli/parser.py:main()` |
| **Dispatch Pattern** | `set_defaults(func=...)` يربط تلقائياً |
| **YAGNI** | لا Facade، لا __main__، لا تعقيد زائد |

---

## 🧪 التحقق من عمل الـ CLI المُحدّث

### 0️⃣ أولاً: أعد التثبيت (مرة واحدة عند تغيير `pyproject.toml`)

```bash
pip install -e .
```

---

### 1️⃣ شاشة المساعدة وقائمة الأوامر

**عبر أمر النظام:**
```bash
forex --help
```

**أو عبر التشغيل اليدوي:**
```bash
python main.py --help
```

**المتوقع:**
```
usage: forex [-h] {version,status} ...

Forex Tracker - CLI Currency Analysis Tool

Commands:
  {version,status}
    version         Show current application version
    status          Check system and configuration status
```

---

### 2️⃣ تجربة أمر `version`

```bash
forex version
# أو
python main.py version
```

**المتوقع:**
```
Forex Tracker v0.1.0
```

---

### 3️⃣ تجربة أمر `status`

```bash
forex status
# أو
python main.py status
```

**المتوقع:**
```
--- Forex Tracker Status ---
App: Forex Tracker
Log File: ./logs/app.log
Data Dir: ./data
```

مع تسجيل الحدث في الـ Logger:
```
[timestamp] [INFO] [forex_tracker]: CLI status command executed successfully.
```

---

### 4️⃣ تجربة تشغيل بدون أوامر

```bash
forex
# أو
python main.py
```

**المتوقع:** عرض شاشة الـ Help تلقائياً والخروج بـ `exit code = 0`.

---

## ⚠️ ملاحظة مهمة حول التوافق

بعد التعديل، **المساران متوافقان تماماً** بدون أي تضارب:

| الطريقة | الأمر | نقطة الدخول الفعلية |
|---|---|---|
| **أمر النظام** | `forex --help` | `cli.parser:main` |
| **التشغيل اليدوي** | `python main.py --help` | `main.py` → `cli.parser.main` |
| **التشغيل المباشر للموديول** | `python -m cli.parser --help` | `cli.parser:__main__` |

كلهم في النهاية بينادوا **نفس الدالة** `main()` في `cli/parser.py` — فلا تكرار، ولا اختلاف في السلوك. ✅

In [4]:
%run main.py --help
%run main.py version
%run main.py status

[2026-10-11 14:32:51] [INFO] [forex_tracker]: CLI status command executed successfully.


usage: forex [-h] {version,status} ...

Forex Tracker CLI Analysis Tool

options:
  -h, --help        show this help message and exit

Commands:
  Available Cli Action

  {version,status}
    version         Show current application version
    status          Check system status and basic settings
Forex Tracker v0.1.0
--- Forex Tracker Status ---
App: Forex Tracker
Log File: ./logs/app.log
Data Dir: ./data


In [5]:
import sys
sys.argv = ["forex", "version"]   # محاكاة الأمر

from cli.parser import main
main()

Forex Tracker v0.1.0


In [6]:
import sys
sys.argv = ["forex", "status"]

# إعادة تحميل الموديول لو سبق تحميله
import importlib
import cli.parser
importlib.reload(cli.parser)

cli.parser.main()

[2026-10-11 14:33:30] [INFO] [forex_tracker]: CLI status command executed successfully.


--- Forex Tracker Status ---
App: Forex Tracker
Log File: ./logs/app.log
Data Dir: ./data


In [7]:
%who

Namespace	 cli	 fake_args	 importlib	 logger	 main	 sys	 
